# Sports Market Compass: Coverage Gap EDA

**Author:** Alessandro Attene  
**Type:** Exploratory Data Analysis (EDA)  

---

## Business Questions

A sports management platform operating in Italy wants to understand where its biggest growth opportunities lie. This analysis answers five key questions:

1. **Where should the platform expand first?** Which provinces show the largest gap between registered sports entities and platform coverage?
2. **Which sports represent the biggest untapped opportunity?** What is the sport mix on the platform, and which areas are under-represented?
3. **How is the platform growing over time?** What does the registration trajectory look like?
4. **What should a phased expansion strategy look like?** How can provinces be prioritized based on market size and current gap?
5. **What is the total addressable market not yet reached?** How large is the coverage gap in absolute terms?

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# The notebook lives in notebooks/: the repository root is one level up.
PROJECT_ROOT = Path("..").resolve()

# Consistent style
sns.set_theme(style="whitegrid", font_scale=1.1)

# Extra breathing room between the plot area and titles/axis labels: seaborn
# defaults keep them too close on dense charts. Must come after set_theme,
# which would otherwise reset these values.
plt.rcParams["axes.titlepad"] = 16
plt.rcParams["axes.labelpad"] = 10

PALETTE = sns.color_palette("tab10")
COLOR_PRIMARY = "#2563EB"
COLOR_SECONDARY = "#F59E0B"
COLOR_GAP = "#DC2626"
COLOR_COVERED = "#16A34A"

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Derived CSVs land next to the data, never in the repository: data/ is
# gitignored, whichever source (real or sample) fed the run.
ANALYSIS_DIR = PROJECT_ROOT / "data" / "analysis"

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:  {FIGURES_DIR}")

In [ ]:
# --- Figure localization (EN canonical + IT) ---------------------------------
# The repository is English-first, but the project targets the Italian market:
# every figure is rendered twice: English to reports/figures/ (embedded by
# README.md and reports/REPORT.md) and Italian to reports/figures/it/
# (embedded by it/README.md and reports/it/REPORT.md).
# Only chart furniture (titles, axis labels, legends) is localized; data-derived
# tick labels keep their source values, except sport areas, which
# get a display-only mapping so Italian charts read naturally for stakeholders.

LANGS: tuple[str, ...] = ("en", "it")

FIG_DIRS: dict[str, Path] = {"en": FIGURES_DIR, "it": FIGURES_DIR / "it"}
for _dir in FIG_DIRS.values():
    _dir.mkdir(parents=True, exist_ok=True)

LABELS: dict[str, dict[str, str]] = {
    "en": {
        "platform_entities": "Platform Entities",
        "platform_region_title": "Platform Presence by Region",
        "entities_per_province": "Entities per Province",
        "n_provinces": "Number of Provinces",
        "province_dist_title": "Distribution of Platform Entities per Province",
        "province_spread_title": "Province-Level Spread (Top 10 Regions)",
        "median": "Median",
        "covered_label": "Platform covered",
        "gap_label": "Coverage gap",
        "n_sport_entities": "Number of Sports Entities",
        "gap_region_title": "Coverage Gap by Region: Platform vs Total Market",
        "full_coverage_line": "100% coverage line",
        "coverage_ratio": "Coverage Ratio",
        "registry_entities_axis": "Registry Entities (Total Market)",
        "platform_entities_axis": "Platform Entities (Current Coverage)",
        "scatter_title": "Registry vs Platform: Province-Level Coverage",
        "ratio_dist_title": "Distribution of Coverage Ratio",
        "gap_axis": "Coverage Gap (entities)",
        "top_gap_title": "Top 15 Provinces by Coverage Gap",
        "sport_entity_pairs": "Entity-Area Pairs",
        "sport_mix_title": "Sport Mix by Area",
        "sport_concentration_title": "Sport Concentration",
        "other": "Other",
        "heatmap_title": "Platform Presence: Sport Area x Region",
        "registration_year": "Registration Year",
        "new_registrations": "New Registrations",
        "cumulative_entities": "Cumulative Entities",
        "cumulative_label": "Cumulative",
        "growth_title": "Platform Growth: Entity Registrations Over Time",
        "no_year_tick": "N/A",
        "no_year_label": "No registration year",
        "market_size_axis": "Market Size (Registry Entities)",
        "gap_unreached_axis": "Coverage Gap (Unreached Entities)",
        "priority_matrix_title": "Expansion Priority Matrix",
        "priority_tier": "Priority Tier",
        "high_priority": "HIGH PRIORITY",
        "low_priority": "LOW PRIORITY",
        "sport_opp_xlabel": "Number of Top-10 Areas with < 10 Entities (out of {n})",
        "sport_opp_title": "Sport Opportunity Index by Region",
        "choropleth_title": "Platform Coverage Across Italian Provinces",
        "no_data": "No data",
    },
    "it": {
        "platform_entities": "Società in piattaforma",
        "platform_region_title": "Presenza della piattaforma per regione",
        "entities_per_province": "Società per provincia",
        "n_provinces": "Numero di province",
        "province_dist_title": "Distribuzione delle società in piattaforma per provincia",
        "province_spread_title": "Dispersione a livello provinciale (Top 10 regioni)",
        "median": "Mediana",
        "covered_label": "Coperto dalla piattaforma",
        "gap_label": "Gap di copertura",
        "n_sport_entities": "Numero di società sportive",
        "gap_region_title": "Gap di copertura per regione: piattaforma vs mercato totale",
        "full_coverage_line": "Linea di copertura al 100%",
        "coverage_ratio": "Tasso di copertura",
        "registry_entities_axis": "Società nel registro (mercato totale)",
        "platform_entities_axis": "Società in piattaforma (copertura attuale)",
        "scatter_title": "Registro vs piattaforma: copertura a livello provinciale",
        "ratio_dist_title": "Distribuzione del tasso di copertura",
        "gap_axis": "Gap di copertura (società)",
        "top_gap_title": "Top 15 province per gap di copertura",
        "sport_entity_pairs": "Coppie società-area",
        "sport_mix_title": "Mix sportivo per area",
        "sport_concentration_title": "Concentrazione sportiva",
        "other": "Altro",
        "heatmap_title": "Presenza in piattaforma: area sportiva x regione",
        "registration_year": "Anno di registrazione",
        "new_registrations": "Nuove registrazioni",
        "cumulative_entities": "Società cumulate",
        "cumulative_label": "Cumulato",
        "growth_title": "Crescita della piattaforma: registrazioni nel tempo",
        "no_year_tick": "N/D",
        "no_year_label": "Senza anno di registrazione",
        "market_size_axis": "Dimensione del mercato (società nel registro)",
        "gap_unreached_axis": "Gap di copertura (società non raggiunte)",
        "priority_matrix_title": "Matrice di priorità di espansione",
        "priority_tier": "Livello di priorità",
        "high_priority": "PRIORITÀ ALTA",
        "low_priority": "PRIORITÀ BASSA",
        "sport_opp_xlabel": "Numero di aree top-10 con < 10 società (su {n})",
        "sport_opp_title": "Indice di opportunità sportiva per regione",
        "choropleth_title": "Copertura della piattaforma nelle province italiane",
        "no_data": "Nessun dato",
    },
}

# Display-only mapping for sport areas: source values stay English
# in the data (stable identifiers used by CSV outputs and the EN docs);
# Italian charts rename them at render time. EN uses an identity mapping.
MACRO_DISPLAY: dict[str, dict[str, str]] = {
    "en": {},
    "it": {
        "Football": "Calcio",
        "Basketball": "Pallacanestro",
        "Volleyball": "Pallavolo",
        "Tennis & Racquet Sports": "Tennis e racchette",
        "Water & Nautical Sports": "Sport acquatici e nautici",
        "Cycling & Motors": "Ciclismo e motori",
        "Athletics & Endurance": "Atletica ed endurance",
        "Gymnastics & Dance": "Ginnastica e danza",
        "Martial Arts & Combat": "Arti marziali e combattimento",
        "Fitness & Wellness": "Fitness e benessere",
        "Winter & Mountain Sports": "Sport invernali e montagna",
        "Skating & Rollersports": "Rotellistica",
        "Team Sports (Other)": "Altri sport di squadra",
        "Precision & Target Sports": "Tiro e precisione",
        "Equestrian & Dog Sports": "Sport equestri e cinofilia",
        "Other Activities": "Altre attività",
    },
}

# Priority tier identifiers: defined once and reused everywhere they are
# needed (qcut labels, display mapping, chart colors), so data, figures and
# exports can never drift apart.
TIER_1: str = "Tier 1"
TIER_2: str = "Tier 2"
TIER_3: str = "Tier 3"
TIER_4: str = "Tier 4"

# Display-only mapping for priority tiers: like MACRO_DISPLAY, the source
# values stay English in the data ("Tier n" is the stable identifier written
# to expansion_priority_by_province.csv); Italian charts rename them at
# render time ("Livello n"). EN uses an identity mapping.
TIER_DISPLAY: dict[str, dict[str, str]] = {
    "en": {},
    "it": {
        TIER_1: "Livello 1",
        TIER_2: "Livello 2",
        TIER_3: "Livello 3",
        TIER_4: "Livello 4",
    },
}


def save_and_show(fig: plt.Figure, filename: str, lang: str) -> None:
    """Save the figure to its language folder; display only the EN (canonical) one."""
    fig.savefig(FIG_DIRS[lang] / filename, dpi=150, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close(fig)


print("Figure output dirs:")
for _lang, _dir in FIG_DIRS.items():
    print(f"  [{_lang}] {_dir}")

### Data Sources

| Source | Description | Granularity |
|--------|-------------|-------------|
| **Registry** | Official sports registry: total registered entities by province | Province (107) |
| **Platform** | Sports management platform: entities currently on the platform | Province + Sport area + Year |

> **Privacy note:** Platform raw data has been sanitized at collection time. Only `sport`, `registration_year`, `province_abbr`, and `region_code` are retained; `sport` carries **area-level values** (16 federation-inspired areas, assigned upstream by the collection pipeline). No personal data (names, addresses, coordinates) is stored.

> **Data provenance:** this repository ships no real dataset. At runtime the notebook loads `data/` (your own collection, gitignored) when present, and falls back to the committed synthetic `data_sample/` otherwise. Published figures are rendered locally from a real collection; the sample only keeps the analysis reproducible.

---
## 1. Data Loading & Preparation

In [ ]:
# --- Data source resolution ---------------------------------------------------
# Real data wins, the committed sample keeps the notebook runnable: a fresh
# clone has no data/ (gitignored) and falls back to data_sample/, whose
# values are invented; only its geography is real.
DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_DIR = PROJECT_ROOT / "data_sample"

_counts_files = (
    "registry_entity_counts_by_province.csv",
    "platform_entity_counts_by_province.csv",
)

if all((DATA_DIR / name).exists() for name in _counts_files):
    ACTIVE_DATA_DIR = DATA_DIR
    print("Data source: data/ (real collected data)")
else:
    ACTIVE_DATA_DIR = SAMPLE_DIR
    print("Data source: data_sample/ (synthetic sample, invented values)")

REGISTRY_COUNTS_CSV = ACTIVE_DATA_DIR / "registry_entity_counts_by_province.csv"
PLATFORM_COUNTS_CSV = ACTIVE_DATA_DIR / "platform_entity_counts_by_province.csv"
PLATFORM_ENTITIES_JSON = ACTIVE_DATA_DIR / "platform_entities.json"

_missing = [p for p in (REGISTRY_COUNTS_CSV, PLATFORM_COUNTS_CSV) if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "Required data files not found. Regenerate the sample with\n"
        "  python scripts/generate_data_sample.py\n"
        "or place your own collection in data/. Missing:\n"
        + "\n".join(f"  - {p}" for p in _missing)
    )
print("Data files found. Ready to load.")

In [ ]:
# Registry data: entity counts by province
# keep_default_na: Napoli's abbreviation is literally "NA", which pandas
# would otherwise parse as missing, breaking merges, labels and the map.
df_registry = pd.read_csv(REGISTRY_COUNTS_CSV, keep_default_na=False, na_values=[""])
print(
    f"Registry: {df_registry.shape[0]} provinces, {df_registry['entities_total'].sum():,} entities"
)
df_registry.head()

In [ ]:
# Platform data: entity counts by province
# Same guard as the registry read: "NA" (Napoli) must stay a string.
df_platform = pd.read_csv(PLATFORM_COUNTS_CSV, keep_default_na=False, na_values=[""])
print(
    f"Platform: {df_platform.shape[0]} provinces, {df_platform['platform_entities'].sum():,} entities"
)
df_platform.head()

In [ ]:
# Load platform entities (sanitized export) for area-level and temporal analysis
if PLATFORM_ENTITIES_JSON.exists():
    with open(PLATFORM_ENTITIES_JSON, encoding="utf-8") as f:
        raw_data = json.load(f)

    df_entities = pd.DataFrame(raw_data["items"])
    print(f"Platform entities: {len(df_entities):,}")
    print(
        f"Multi-area entities: {df_entities['sport'].apply(len).gt(1).sum():,} ({df_entities['sport'].apply(len).gt(1).mean():.0%})"
    )

    # Explode the area list: one row per (entity, area) pair
    df_exploded = df_entities.explode("sport").reset_index(drop=True)
    print(f"Entity-area pairs after explode: {len(df_exploded):,}")
    print(f"Distinct sport areas: {df_exploded['sport'].nunique()}")
else:
    print(f"Entities export not found at {PLATFORM_ENTITIES_JSON}.")
    print(
        "Area-level and temporal analyses will be skipped.\n"
        "Place the sanitized entities export next to the counts CSVs to enable them."
    )
    # Empty DataFrames so downstream cells do not raise NameError
    df_entities = pd.DataFrame(
        columns=["sport", "registration_year", "province_abbr", "region_code"]
    )
    df_exploded = pd.DataFrame(
        columns=[
            "sport",
            "registration_year",
            "province_abbr",
            "region_code",
            "region_name",
            "sport_macro",
        ]
    )

In [ ]:
# Sport dimension: areas assigned upstream by the collection pipeline
# The published sport dimension is area-level: the fine-grained sport labels
# observed at the source are grouped into 16 federation-inspired areas
# before the data reaches this repository: same pattern as the province
# harmonization, applied once upstream and documented here. The taxonomy
# travels with the payload ("sport_areas" in the JSON envelope); an entity
# offering several sports of the same area counts once for that area.
df_exploded["sport_macro"] = df_exploded["sport"]

if df_exploded.empty:
    print("No entity-level data: sport-area sections will be skipped.")
else:
    print(f"Sport areas observed: {df_exploded['sport_macro'].nunique()}")
    print("\nEntity-area pairs by area:")
    print(df_exploded["sport_macro"].value_counts().to_string())

In [ ]:
# --- Geography harmonization (methodology) ------------------------------------
# Canonical region names for every code observed in the data. The platform
# source emits "VAO" for Valle d'Aosta instead of the standard "VDA": both
# alias the same canonical name.
REGION_CODE_TO_NAME: dict[str, str] = {
    "ABR": "Abruzzo",
    "BAS": "Basilicata",
    "CAL": "Calabria",
    "CAM": "Campania",
    "EMR": "Emilia-Romagna",
    "FVG": "Friuli-Venezia Giulia",
    "LAZ": "Lazio",
    "LIG": "Liguria",
    "LOM": "Lombardia",
    "MAR": "Marche",
    "MOL": "Molise",
    "PIE": "Piemonte",
    "PUG": "Puglia",
    "SAR": "Sardegna",
    "SIC": "Sicilia",
    "TOS": "Toscana",
    "TAA": "Trentino-Alto Adige/Südtirol",
    "UMB": "Umbria",
    "VAO": "Valle d'Aosta/Vallée d'Aoste",
    "VDA": "Valle d'Aosta/Vallée d'Aoste",
    "VEN": "Veneto",
}

# The platform keeps the province code each entity was registered under, so
# codes abolished by the 2016 Sardinian reform (CI, OG, OT) survive next to
# current ones: SU and CI could never coexist in a real layout. Entities are
# remapped to the successor provinces, i.e. to the registry layout at the
# snapshot date. Known limits, to be stated with any conclusion:
# - codes that survived 2016 (notably CA) cannot be re-verified below
#   province level: province is the finest granularity collected (privacy
#   by design), so pre-2016 entities now in SU territory may stay under CA;
# - the incoming province reform will reassign SU/VS/OG/OT to new provinces:
#   this map is versioned and must be revisited when the sources adopt the
#   new layout (no VS entry today for the same reason: post-reform VS is a
#   valid current code).
PROVINCE_ABBR_HARMONIZATION: dict[str, str] = {"CI": "SU", "OG": "NU", "OT": "SS"}

# Counts CSVs arrive already harmonized from the collection pipeline; the
# entity-level export keeps source values by design and is aligned here.
# Both operations are idempotent on already-clean inputs.
df_platform["region_name"] = (
    df_platform["region_code"]
    .map(REGION_CODE_TO_NAME)
    .fillna(df_platform["region_name"])
)

for _frame in (df_entities, df_exploded):
    if not _frame.empty:
        _frame["province_abbr"] = _frame["province_abbr"].replace(
            PROVINCE_ABBR_HARMONIZATION
        )
df_exploded["region_name"] = (
    df_exploded["region_code"].map(REGION_CODE_TO_NAME).fillna(df_exploded["region_code"])
)

# Merge registry + platform at province level
df = df_registry.merge(
    df_platform[["province_abbr", "platform_entities"]],
    on="province_abbr",
    how="left",
)
df["platform_entities"] = df["platform_entities"].fillna(0).astype(int)
df["coverage_gap"] = df["entities_total"] - df["platform_entities"]
df["coverage_ratio"] = df["platform_entities"] / df["entities_total"]

registry_provinces = len(df_registry)
total_provinces = 107

print(
    f"Merged dataset: {len(df)} provinces (registry coverage: {registry_provinces}/{total_provinces})"
)
print(f"Total registry entities:  {df['entities_total'].sum():,}")
print(f"Total platform entities:  {df['platform_entities'].sum():,}")
print(f"Total coverage gap:       {df['coverage_gap'].sum():,}")

if registry_provinces < total_provinces:
    print(
        f"\n⚠ Registry data covers {registry_provinces} of {total_provinces} provinces."
    )
    print("  Provide the full registry counts dataset for a complete gap analysis.")
    print(
        "  Platform-only analyses (sport, temporal, geographic distribution) are unaffected."
    )

---
## 2. Data Quality Overview

In [ ]:
print("=== Registry ===")
display(df_registry.describe())
print(f"Null values: {df_registry.isna().sum().sum()}")

print("\n=== Platform (province-level) ===")
display(df_platform.describe())
print(f"Null values: {df_platform.isna().sum().sum()}")

print("\n=== Platform (entity-level) ===")
total_entities = len(df_entities)
if total_entities > 0:
    with_year = df_entities["registration_year"].notna().sum()
    print(f"Entities: {total_entities:,}")
    print(f"With registration_year: {with_year:,} ({with_year/total_entities:.0%})")
    print(f"Without registration_year: {total_entities - with_year:,}")
    print(
        f"Year range: {df_entities['registration_year'].min():.0f} - {df_entities['registration_year'].max():.0f}"
    )
    print(f"Unique provinces: {df_entities['province_abbr'].nunique()}")
    print(f"Unique regions: {df_entities['region_code'].nunique()}")
else:
    print("No entity-level data available (raw platform payload not found).")

> **Note on entity-area pairs:** the sport dimension is published at area granularity (16 federation-inspired areas, see the methodology cell above), and many entities span multiple areas. When analyzing by area, each entity is counted once per area it covers: area-level totals therefore represent _entity-area pairs_, not unique entities.

---
## 3. Platform Geographic Distribution

Before analyzing the coverage gap (which requires registry data), let's understand where the platform currently operates across Italy's 107 provinces.

In [ ]:
# Platform entities by region
df_plat_region = (
    df_platform.groupby("region_name", as_index=False)
    .agg(
        platform_total=("platform_entities", "sum"),
        province_count=("province_abbr", "count"),
    )
    .sort_values("platform_total", ascending=True)
)

for lang in LANGS:
    L = LABELS[lang]
    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(
        df_plat_region["region_name"],
        df_plat_region["platform_total"],
        color=COLOR_PRIMARY,
    )

    for bar, val in zip(bars, df_plat_region["platform_total"]):
        ax.text(
            bar.get_width() + 8,
            bar.get_y() + bar.get_height() / 2,
            f"{val:,}",
            va="center",
            fontsize=9,
        )

    ax.set_xlabel(L["platform_entities"])
    ax.set_title(L["platform_region_title"], fontsize=14, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    plt.tight_layout()
    save_and_show(fig, "platform_distribution_by_region.png", lang)

In [ ]:
# Distribution of platform entities per province
top_regions = df_plat_region.nlargest(10, "platform_total")["region_name"].tolist()
df_top = df_platform[df_platform["region_name"].isin(top_regions)]
region_order = (
    df_top.groupby("region_name")["platform_entities"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

for lang in LANGS:
    L = LABELS[lang]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    axes[0].hist(
        df_platform["platform_entities"],
        bins=30,
        color=COLOR_PRIMARY,
        edgecolor="white",
    )
    axes[0].set_xlabel(L["entities_per_province"])
    axes[0].set_ylabel(L["n_provinces"])
    axes[0].set_title(L["province_dist_title"])
    axes[0].axvline(
        df_platform["platform_entities"].median(),
        color=COLOR_GAP,
        linestyle="--",
        label=f"{L['median']}: {df_platform['platform_entities'].median():.0f}",
    )
    axes[0].legend()

    # Box plot by region (top 10 regions)
    sns.boxplot(
        data=df_top,
        y="region_name",
        x="platform_entities",
        order=region_order,
        ax=axes[1],
        color=COLOR_PRIMARY,
    )
    axes[1].set_xlabel(L["entities_per_province"])
    axes[1].set_ylabel("")
    axes[1].set_title(L["province_spread_title"])

    plt.tight_layout()
    save_and_show(fig, "platform_province_distribution.png", lang)

**Key finding:** In the current dataset extract, the platform is present across the provinces included in the data snapshot, but concentration is highly uneven. Lombardia alone accounts for a disproportionate share of total entities, and within each region the distribution is heavily skewed toward the regional capital.

---
## 4. Coverage Gap Analysis

Comparing registry data (total addressable market) with platform presence to quantify the gap.

| KPI | Formula | Interpretation |
|-----|---------|----------------|
| **Coverage Ratio** | `platform_entities / entities_total` | 0 = no coverage, 1 = full coverage |
| **Coverage Gap** | `entities_total - platform_entities` | Absolute number of unreached entities |

In [ ]:
# Coverage Gap by Region (stacked: covered + gap)
df_region = df.groupby("region_name", as_index=False).agg(
    registry_total=("entities_total", "sum"),
    platform_total=("platform_entities", "sum"),
)
df_region["coverage_gap"] = df_region["registry_total"] - df_region["platform_total"]
df_region["coverage_ratio"] = df_region["platform_total"] / df_region["registry_total"]
df_region = df_region.sort_values("registry_total", ascending=True)

for lang in LANGS:
    L = LABELS[lang]
    fig, ax = plt.subplots(figsize=(12, 7))

    ax.barh(
        df_region["region_name"],
        df_region["platform_total"],
        color=COLOR_COVERED,
        label=L["covered_label"],
    )
    ax.barh(
        df_region["region_name"],
        df_region["coverage_gap"],
        left=df_region["platform_total"],
        color=COLOR_GAP,
        alpha=0.7,
        label=L["gap_label"],
    )

    # Annotate with ratio text on the right
    for idx, (_, row) in enumerate(df_region.iterrows()):
        ax.text(
            row["registry_total"] + 30,
            idx,
            f"{row['coverage_ratio']:.0%}",
            va="center",
            fontsize=8,
            color="gray",
        )

    ax.set_xlabel(L["n_sport_entities"])
    ax.set_title(L["gap_region_title"], fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    plt.tight_layout()
    save_and_show(fig, "coverage_gap_by_region_stacked.png", lang)

In [ ]:
# Scatter: Registry entities vs Platform entities per province
max_val = max(df["entities_total"].max(), df["platform_entities"].max()) * 1.1
top5 = df.nlargest(5, "coverage_gap")

for lang in LANGS:
    L = LABELS[lang]
    fig, ax = plt.subplots(figsize=(10, 8))

    ax.plot(
        [0, max_val],
        [0, max_val],
        "--",
        color="gray",
        alpha=0.5,
        label=L["full_coverage_line"],
    )

    scatter = ax.scatter(
        df["entities_total"],
        df["platform_entities"],
        c=df["coverage_ratio"],
        cmap="RdYlGn",
        s=80,
        edgecolors="white",
        linewidth=0.5,
        vmin=0,
        vmax=0.5,
        zorder=5,
    )
    plt.colorbar(scatter, ax=ax, label=L["coverage_ratio"], shrink=0.8)

    # Annotate top-5 by gap
    for _, row in top5.iterrows():
        ax.annotate(
            f"{row['province_name']} ({row['province_abbr']})",
            (row["entities_total"], row["platform_entities"]),
            textcoords="offset points",
            xytext=(8, 8),
            fontsize=8,
            arrowprops={"arrowstyle": "->", "color": "gray", "lw": 0.8},
        )

    ax.set_xlabel(L["registry_entities_axis"])
    ax.set_ylabel(L["platform_entities_axis"])
    ax.set_title(L["scatter_title"], fontsize=14, fontweight="bold")
    ax.legend(loc="upper left")

    plt.tight_layout()
    save_and_show(fig, "scatter_registry_vs_platform.png", lang)

In [ ]:
# Coverage ratio distribution
df_top_gap = df.nlargest(15, "coverage_gap")

for lang in LANGS:
    L = LABELS[lang]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of coverage ratio
    axes[0].hist(df["coverage_ratio"], bins=20, color=COLOR_PRIMARY, edgecolor="white")
    axes[0].set_xlabel(L["coverage_ratio"])
    axes[0].set_ylabel(L["n_provinces"])
    axes[0].set_title(L["ratio_dist_title"])
    axes[0].axvline(
        df["coverage_ratio"].median(),
        color=COLOR_GAP,
        linestyle="--",
        label=f"{L['median']}: {df['coverage_ratio'].median():.2%}",
    )
    axes[0].legend()

    # Top 15 provinces by coverage gap
    ax2 = axes[1]
    ax2.barh(
        df_top_gap["province_name"] + " (" + df_top_gap["province_abbr"] + ")",
        df_top_gap["coverage_gap"],
        color=COLOR_GAP,
        alpha=0.8,
    )
    ax2.set_xlabel(L["gap_axis"])
    ax2.set_title(L["top_gap_title"])
    ax2.invert_yaxis()

    plt.tight_layout()
    save_and_show(fig, "coverage_ratio_distribution.png", lang)

**Key findings:**
- The vast majority of provinces show a **very low coverage ratio**, indicating the platform has reached only a small fraction of the total addressable market.
- The largest absolute gaps are concentrated in the most populated provinces.
- The scatter plot confirms that points cluster near the x-axis, far below the 100% coverage diagonal, highlighting a market largely untapped.

---
## 5. Sport-Area Analysis

The sport dimension arrives at area granularity: 16 federation-inspired areas, grouped upstream by the collection pipeline (see the methodology note above).

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport mix distribution chart.")
else:
    # Sport area distribution
    macro_counts = df_exploded["sport_macro"].value_counts()
    macro_pct = (macro_counts / macro_counts.sum() * 100).round(1)
    top_n = 8
    colors = sns.color_palette("tab10", n_colors=len(macro_counts))

    for lang in LANGS:
        L = LABELS[lang]
        # Rename data-derived category labels for display only (identity for EN)
        m_counts = macro_counts.rename(index=MACRO_DISPLAY[lang])
        m_pct = macro_pct.rename(index=MACRO_DISPLAY[lang])

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Bar chart
        bars = axes[0].barh(
            m_counts.index[::-1], m_counts.values[::-1], color=colors[::-1]
        )
        for bar, val, pct in zip(bars, m_counts.values[::-1], m_pct.values[::-1]):
            axes[0].text(
                bar.get_width() + 5,
                bar.get_y() + bar.get_height() / 2,
                f"{val:,} ({pct}%)",
                va="center",
                fontsize=9,
            )
        axes[0].set_xlabel(L["sport_entity_pairs"])
        axes[0].set_title(L["sport_mix_title"], fontsize=13, fontweight="bold")

        # Pie chart for top categories
        top_macro = m_counts.head(top_n)
        other = pd.Series({L["other"]: m_counts.iloc[top_n:].sum()})
        pie_data = pd.concat([top_macro, other])
        pie_colors = colors[:top_n] + [(0.8, 0.8, 0.8)]
        axes[1].pie(
            pie_data.values,
            labels=pie_data.index,
            autopct="%1.1f%%",
            colors=pie_colors,
            startangle=90,
            textprops={"fontsize": 9},
        )
        axes[1].set_title(
            L["sport_concentration_title"], fontsize=13, fontweight="bold"
        )

        plt.tight_layout()
        save_and_show(fig, "sport_mix_distribution.png", lang)

    top3_share = macro_pct.iloc[:3].sum()
    print(
        f"Top 3 areas account for {top3_share:.1f}% of all entity-area pairs."
    )

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport × region heatmap.")
else:
    # Sport x Region heatmap (areas)
    pivot_sport = df_exploded.pivot_table(
        index="region_name",
        columns="sport_macro",
        values="sport",
        aggfunc="count",
        fill_value=0,
    )

    # Sort regions by total platform presence
    pivot_sport = pivot_sport.loc[
        pivot_sport.sum(axis=1).sort_values(ascending=False).index
    ]

    for lang in LANGS:
        L = LABELS[lang]
        # Rename data-derived category labels for display only (identity for EN)
        pivot_display = pivot_sport.rename(columns=MACRO_DISPLAY[lang])

        fig, ax = plt.subplots(figsize=(16, 9))
        sns.heatmap(
            pivot_display,
            cmap="YlOrRd",
            linewidths=0.5,
            ax=ax,
            fmt=",d",
            annot=True,
            annot_kws={"size": 8},
            cbar_kws={"label": L["sport_entity_pairs"]},
        )
        ax.set_title(L["heatmap_title"], fontsize=14, fontweight="bold")
        ax.set_ylabel("")
        ax.set_xlabel("")
        plt.xticks(rotation=35, ha="right")

        plt.tight_layout()
        save_and_show(fig, "sport_region_heatmap.png", lang)

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport diversity index.")
else:
    # Sport diversity index: how many distinct areas per region
    diversity = (
        df_exploded.groupby("region_name")["sport_macro"]
        .nunique()
        .sort_values(ascending=False)
        .rename("sport_areas")
    )

    print("Sport Diversity Index (areas per region):")
    print(f"  Max: {diversity.max()} areas ({diversity.idxmax()})")
    print(f"  Min: {diversity.min()} areas ({diversity.idxmin()})")
    print(f"  Mean: {diversity.mean():.1f}")
    print()
    display(diversity.to_frame().T)

**Key findings:**
- **Football dominates** the platform, representing the largest share of entity-area pairs. Together with the next two areas, they account for over half of all activity.
- The heatmap reveals a **strong geographic concentration**: Lombardia, Lazio, and Veneto lead across almost every sport area.
- Sport diversity is relatively uniform across regions (most regions cover 10+ areas), but the _depth_ (number of entities per area) varies dramatically.
- **Under-represented areas** like Athletics & Endurance, Winter & Mountain Sports, and Water & Nautical Sports could represent niche expansion opportunities in regions with relevant infrastructure.

---
## 6. Growth Trajectory

Analyzing registration year data to understand the platform's growth pattern.

In [ ]:
if len(df_entities) > 0:
    # Registration year trend
    df_year = df_entities.dropna(subset=["registration_year"]).copy()
    df_year["registration_year"] = df_year["registration_year"].astype(int)

    year_counts = df_year["registration_year"].value_counts().sort_index()
    cumulative = year_counts.cumsum()
    year_labels = year_counts.index.astype(str).tolist()

    # Entities without a registration year are shown as a separate, clearly
    # labeled bar next to the time series instead of disappearing silently.
    # They stay out of the cumulative line: assigning them a year (or a
    # position in the series) would fabricate data the source does not carry.
    no_year = len(df_entities) - len(df_year)

    for lang in LANGS:
        L = LABELS[lang]
        fig, ax1 = plt.subplots(figsize=(12, 6))

        if no_year:
            nd_bars = ax1.bar(
                [L["no_year_tick"]],
                [no_year],
                color="#9CA3AF",
                label=L["no_year_label"],
                zorder=3,
            )
        else:
            nd_bars = []

        bars = ax1.bar(
            year_labels,
            year_counts.values,
            color=COLOR_PRIMARY,
            label=L["new_registrations"],
            zorder=3,
        )

        all_bars = list(nd_bars) + list(bars)
        all_vals = ([no_year] if no_year else []) + year_counts.values.tolist()
        for bar, val in zip(all_bars, all_vals):
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 15,
                f"{val:,}",
                ha="center",
                fontsize=10,
                fontweight="bold",
            )

        ax1.set_xlabel(L["registration_year"])
        ax1.set_ylabel(L["new_registrations"], color=COLOR_PRIMARY)
        ax1.tick_params(axis="y", labelcolor=COLOR_PRIMARY)

        # Cumulative line on secondary axis: dated series only (the no-year
        # bar stays out of it)
        ax2 = ax1.twinx()
        ax2.plot(
            year_labels,
            cumulative.values,
            color=COLOR_SECONDARY,
            marker="o",
            linewidth=2.5,
            label=L["cumulative_label"],
            zorder=4,
        )
        ax2.set_ylabel(L["cumulative_entities"], color=COLOR_SECONDARY)
        ax2.tick_params(axis="y", labelcolor=COLOR_SECONDARY)

        ax1.set_title(L["growth_title"], fontsize=14, fontweight="bold")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

        plt.tight_layout()
        save_and_show(fig, "registration_trend.png", lang)

    print(
        f"Note: {no_year:,} entities ({no_year/len(df_entities):.0%}) have no "
        "registration year and are shown as the separate no-year bar."
    )
else:
    print("No entity data available; skipping registration year trend analysis.")

**Key finding:** The platform shows a clear growth trajectory. The year-over-year registration volume indicates increasing adoption, though the pace and any seasonal patterns should be monitored to assess market momentum.

---
## 7. Expansion Prioritization Framework

To identify the highest-value provinces for platform expansion, we build a composite scoring model:

| Factor | Weight | Rationale |
|--------|--------|-----------|
| **Gap Score** | 60% | Larger gap = more unreached entities |
| **Density Score** | 40% | Larger market = higher absolute opportunity |

In [ ]:
# Scoring model for province prioritization
df_priority = df.copy()

# Normalize scores to 0-1 range
gap_min, gap_max = df_priority["coverage_gap"].min(), df_priority["coverage_gap"].max()
density_min, density_max = (
    df_priority["entities_total"].min(),
    df_priority["entities_total"].max(),
)

if gap_max > gap_min:
    df_priority["gap_score"] = (df_priority["coverage_gap"] - gap_min) / (
        gap_max - gap_min
    )
else:
    df_priority["gap_score"] = 0.0

if density_max > density_min:
    df_priority["density_score"] = (df_priority["entities_total"] - density_min) / (
        density_max - density_min
    )
else:
    df_priority["density_score"] = 0.0

df_priority["priority_score"] = (
    0.6 * df_priority["gap_score"] + 0.4 * df_priority["density_score"]
)

# Assign tiers by quartile
df_priority["priority_tier"] = pd.qcut(
    df_priority["priority_score"].rank(method="first"),
    q=4,
    labels=[TIER_4, TIER_3, TIER_2, TIER_1],
)

df_priority = df_priority.sort_values("priority_score", ascending=False)

print("Priority tier distribution:")
print(df_priority["priority_tier"].value_counts().sort_index())

In [ ]:
# Priority Matrix: Market Size vs Coverage Gap
tier_colors = {
    TIER_1: "#DC2626",
    TIER_2: "#F59E0B",
    TIER_3: "#3B82F6",
    TIER_4: "#9CA3AF",
}
med_x = df_priority["entities_total"].median()
med_y = df_priority["coverage_gap"].median()

for lang in LANGS:
    L = LABELS[lang]
    fig, ax = plt.subplots(figsize=(12, 9))

    for tier, color in tier_colors.items():
        mask = df_priority["priority_tier"] == tier
        ax.scatter(
            df_priority.loc[mask, "entities_total"],
            df_priority.loc[mask, "coverage_gap"],
            c=color,
            label=TIER_DISPLAY[lang].get(tier, tier),
            s=100,
            edgecolors="white",
            linewidth=0.8,
            zorder=5,
        )

    # Quadrant lines at median
    ax.axvline(med_x, color="gray", linestyle=":", alpha=0.5)
    ax.axhline(med_y, color="gray", linestyle=":", alpha=0.5)

    # Annotate top 10 by priority
    for _, row in df_priority.head(10).iterrows():
        ax.annotate(
            f"{row['province_name']} ({row['province_abbr']})",
            (row["entities_total"], row["coverage_gap"]),
            textcoords="offset points",
            xytext=(8, 6),
            fontsize=8,
            arrowprops={"arrowstyle": "->", "color": "gray", "lw": 0.7},
        )

    ax.set_xlabel(L["market_size_axis"], fontsize=12)
    ax.set_ylabel(L["gap_unreached_axis"], fontsize=12)
    ax.set_title(L["priority_matrix_title"], fontsize=14, fontweight="bold")
    ax.legend(title=L["priority_tier"], fontsize=10)

    # Quadrant labels
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    ax.text(
        xlim[1] * 0.75,
        ylim[1] * 0.92,
        L["high_priority"],
        fontsize=11,
        color="#DC2626",
        fontweight="bold",
        ha="center",
        alpha=0.6,
    )
    ax.text(
        xlim[0] + (med_x - xlim[0]) * 0.5,
        ylim[0] + (med_y - ylim[0]) * 0.3,
        L["low_priority"],
        fontsize=11,
        color="gray",
        fontweight="bold",
        ha="center",
        alpha=0.5,
    )

    plt.tight_layout()
    save_and_show(fig, "priority_matrix.png", lang)

In [ ]:
# Top 20 provinces for expansion
top20_cols = [
    "region_name",
    "province_name",
    "province_abbr",
    "entities_total",
    "platform_entities",
    "coverage_gap",
    "coverage_ratio",
    "priority_score",
    "priority_tier",
]
print("Top 20 Provinces for Expansion:")
display(
    df_priority.head(20)[top20_cols]
    .reset_index(drop=True)
    .style.format(
        {
            "coverage_ratio": "{:.1%}",
            "priority_score": "{:.3f}",
            "entities_total": "{:,}",
            "platform_entities": "{:,}",
            "coverage_gap": "{:,}",
        }
    )
    .bar(subset=["priority_score"], color=COLOR_GAP, vmin=0, vmax=1)
)

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport opportunity index.")
else:
    # Sport Opportunity by Region: how many of the top-10 areas
    # have fewer than 10 entities in each region (= underserved)
    top10_sports = df_exploded["sport_macro"].value_counts().head(10).index.tolist()

    sport_region_counts = (
        df_exploded[df_exploded["sport_macro"].isin(top10_sports)]
        .groupby(["region_name", "sport_macro"])
        .size()
        .unstack(fill_value=0)
    )

    # Count sports with < 10 entities per region (= opportunity)
    sport_gaps = (sport_region_counts < 10).sum(axis=1).sort_values(ascending=True)

    # Calm sequential palette, intensity scaled on the value: keeps the
    # ranking readable without the alarm effect of a solid warning color.
    max_gap = max(sport_gaps.max(), 1)
    bar_colors = [plt.cm.Blues(0.35 + 0.55 * val / max_gap) for val in sport_gaps.values]

    for lang in LANGS:
        L = LABELS[lang]
        fig, ax = plt.subplots(figsize=(12, 7))
        bars = ax.barh(sport_gaps.index, sport_gaps.values, color=bar_colors)

        for bar, val in zip(bars, sport_gaps.values):
            ax.text(
                bar.get_width() + 0.1,
                bar.get_y() + bar.get_height() / 2,
                f"{val}",
                va="center",
                fontsize=9,
            )

        ax.set_xlabel(L["sport_opp_xlabel"].format(n=len(top10_sports)))
        ax.set_title(L["sport_opp_title"], fontsize=14, fontweight="bold")
        ax.set_xlim(0, len(top10_sports) + 1)

        plt.tight_layout()
        save_and_show(fig, "sport_opportunity_by_region.png", lang)

### Recommended Expansion Strategy

Based on the priority framework, a three-phase approach:

| Phase | Focus | Criteria |
|-------|-------|---------|
| **Phase 1** | Geographic expansion | Tier 1 provinces: largest market and largest gap |
| **Phase 2** | Sport deepening | Add under-represented sport areas in existing regions |
| **Phase 3** | Long-tail expansion | Tier 2 provinces: medium market, moderate gap |

---
## 8. Geographic Visualization

In [ ]:
# Choropleth map of Italy: platform entities by province
try:
    import geopandas as gpd
    import matplotlib.patheffects as pe
    from matplotlib.colors import LinearSegmentedColormap, Normalize, PowerNorm

    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    print("geopandas not installed. Install it to generate the choropleth map:")
    print("  pip install geopandas")

if HAS_GEOPANDAS:
    geo_path = PROJECT_ROOT / "geo" / "provinces.geojson"

    if not geo_path.exists():
        print(f"GeoJSON not found at {geo_path}")
        print("The province boundaries ship with the repository under geo/.")
    else:
        gdf = gpd.read_file(geo_path)

        acr_col = None
        for candidate in ["prov_acr", "sigla", "SIGLA", "prov_sigla", "COD_PROV"]:
            if candidate in gdf.columns:
                acr_col = candidate
                break

        if not acr_col:
            print(
                f"Could not find province abbreviation column. Available: {list(gdf.columns)}"
            )
        else:
            gdf = gdf.merge(
                df_platform[["province_abbr", "platform_entities"]],
                left_on=acr_col,
                right_on="province_abbr",
                how="left",
            )
            # NaNs are kept so that missing_kwds correctly styles unmatched provinces as "No data"

            matched = gdf["platform_entities"].notna().sum()
            if matched == 0:
                print(
                    "Warning: no provinces matched between GeoJSON and platform data: choropleth skipped."
                )
                print(
                    "Check that province abbreviation codes are consistent between both datasets."
                )
            else:
                # Custom colormap: red (low) -> yellow -> green (high)
                cmap_go = LinearSegmentedColormap.from_list(
                    "red_green", ["#DC2626", "#FACC15", "#16A34A"]
                )

                vmin = gdf["platform_entities"].min()
                vmax = gdf["platform_entities"].max()

                if vmin < vmax:
                    # Power normalization: spreads low values across more colors (gamma < 1)
                    norm = PowerNorm(gamma=0.4, vmin=vmin, vmax=vmax)
                else:
                    print(
                        "Warning: all matched platform_entities values are equal: color scale is degenerate."
                    )
                    norm = Normalize(vmin=vmin, vmax=vmin + 1)

                for lang in LANGS:
                    L = LABELS[lang]
                    fig, ax = plt.subplots(1, 1, figsize=(14, 16))
                    gdf.plot(
                        column="platform_entities",
                        ax=ax,
                        legend=True,
                        cmap=cmap_go,
                        norm=norm,
                        edgecolor="gray",
                        linewidth=0.3,
                        legend_kwds={"label": L["platform_entities"], "shrink": 0.6},
                        missing_kwds={"color": "lightgray", "label": L["no_data"]},
                    )

                    # Label each province with its abbreviation at the representative point
                    for _, row in gdf.iterrows():
                        point = row.geometry.representative_point()
                        label = row.get(acr_col, "")
                        if label:
                            ax.annotate(
                                label,
                                xy=(point.x, point.y),
                                ha="center",
                                va="center",
                                fontsize=8,
                                fontweight="bold",
                                color="white",
                                path_effects=[
                                    pe.withStroke(linewidth=2, foreground="black")
                                ],
                            )

                    ax.set_title(L["choropleth_title"], fontsize=16, fontweight="bold")
                    ax.axis("off")

                    plt.tight_layout()
                    save_and_show(fig, "italy_choropleth.png", lang)

                print(f"Matched provinces: {matched} / {len(gdf)}")
                print(f"Entities range: {vmin:.0f} – {vmax:.0f}")
                print(f"Median: {gdf['platform_entities'].median():.0f}")

---
## 9. Conclusions & Recommendations

### Executive Summary

1. **Market sizing:** In the current dataset, the platform is present across all provinces covered in this snapshot (107 in the current extract), but coverage depth varies enormously: a handful of provinces concentrate most of the activity.

2. **Coverage gap:** Where registry data is available, the gap between total registered entities and platform presence is substantial. The platform has captured only a fraction of the total addressable market.

3. **Sport concentration:** Football dominates the platform's sport mix. The top 3 sport areas account for over half of all entity-area pairs, creating both a strength (clear beachhead) and a risk (limited diversification).

4. **Growth trajectory:** Registration data shows sustained growth over recent years, indicating positive market momentum.

5. **Expansion opportunity:** The priority framework identifies high-value provinces where large markets intersect with large gaps: these represent the most impactful targets for geographic expansion.

### Next Steps

- Bring a full registry collection into `data/` to extend the coverage gap analysis to every province in your own snapshot.
- Enrich with demographic data (population, income) for more robust prioritization.
- Monitor registration trends periodically to detect acceleration or slowdown.
- Explore the interactive dashboard on [Looker Studio](https://lookerstudio.google.com/s/tDAIpFPxjls) for drill-down exploration.

In [ ]:
# Export analysis outputs
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Coverage gap by province (merged)
output_gap = ANALYSIS_DIR / "coverage_gap_by_province.csv"
df.to_csv(output_gap, index=False)
print(f"Saved: {output_gap}")

# Expansion priority ranking
output_priority = ANALYSIS_DIR / "expansion_priority_by_province.csv"
df_priority[top20_cols].to_csv(output_priority, index=False)
print(f"Saved: {output_priority}")

# Area-level aggregation by region
output_sport = ANALYSIS_DIR / "platform_sport_by_region.csv"
(
    df_exploded.groupby(["region_name", "sport_macro"])
    .size()
    .reset_index(name="entity_area_pairs")
    .rename(columns={"sport_macro": "sport_area"})
    .to_csv(output_sport, index=False)
)
print(f"Saved: {output_sport}")

print(f"\nAll outputs saved to {ANALYSIS_DIR}")